# Session 43: Permutation Importance and Interpretability Comparison

This notebook trains a 300-tree Random Forest regressor, excludes G1 and G2, computes test-set permutation importance with 10 repeats, and compares it with built-in feature importance. Results are predictive associations, not causal effects.

In [1]:
﻿import json
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder


RANDOM_STATE = 42
N_ESTIMATORS = 300
N_REPEATS = 10


def read_student_data(dataset_path):
    df = pd.read_csv(dataset_path, sep=";")
    if df.shape[1] == 1:
        df = pd.read_csv(dataset_path)

    required = {"G1", "G2", "G3"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(
            "The dataset is missing required columns: " + ", ".join(sorted(missing))
        )

    df = df.drop_duplicates().reset_index(drop=True)
    if df.empty:
        raise ValueError("The dataset contains no usable rows.")
    if df["G3"].isna().any():
        raise ValueError("G3 contains missing values.")
    return df


def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def classify_rank_comparison(rank_difference):
    if rank_difference <= 2:
        return "Strong agreement"
    if rank_difference <= 5:
        return "Moderate disagreement"
    return "Strong disagreement"


def format_feature_list(features):
    return ", ".join(features) if features else "None"


def resolve_repo_root():
    environment_root = os.environ.get("S43_REPO_ROOT")
    if environment_root:
        return Path(environment_root).resolve()

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists():
            return candidate
    return current


def resolve_dataset_path(repo_root):
    environment_dataset = os.environ.get("S43_DATASET")
    if environment_dataset and Path(environment_dataset).is_file():
        return Path(environment_dataset).resolve()

    candidates = [
        repo_root / "student-mat.csv",
        repo_root / "data" / "student-mat.csv",
        repo_root / "data" / "raw" / "student-mat.csv",
        repo_root / "data" / "processed" / "student-mat.csv",
        repo_root / "dataset" / "student-mat.csv",
        Path.home() / "Downloads" / "student-mat.csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    for candidate in repo_root.rglob("student-mat.csv"):
        if not any(part in {".venv", "venv", "node_modules"} for part in candidate.parts):
            return candidate.resolve()

    raise FileNotFoundError(
        "student-mat.csv was not found in the repository or Windows Downloads folder."
    )


def run_analysis():
    repo_root = resolve_repo_root()
    dataset_path = resolve_dataset_path(repo_root)
    notebook_dir = repo_root / "notebooks"
    interpretation_dir = repo_root / "reports" / "interpretability"
    figures_dir = repo_root / "reports" / "figures"

    notebook_dir.mkdir(parents=True, exist_ok=True)
    interpretation_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    df = read_student_data(dataset_path)

    X = df.drop(columns=["G3", "G1", "G2"]).copy()
    y = df["G3"].copy()

    if any(column in X.columns for column in ["G1", "G2", "G3"]):
        raise AssertionError("Leakage check failed: G1, G2, or G3 is present in X.")

    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", numeric_features),
            ("categorical", make_encoder(), categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    Xtr_array = preprocessor.fit_transform(X_train)
    Xte_array = preprocessor.transform(X_test)
    feature_names = preprocessor.get_feature_names_out()

    Xtr_f = pd.DataFrame(Xtr_array, columns=feature_names, index=X_train.index)
    Xte_f = pd.DataFrame(Xte_array, columns=feature_names, index=X_test.index)

    if list(Xtr_f.columns) != list(Xte_f.columns):
        raise AssertionError("Training and test feature columns are not aligned.")

    rf = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(Xtr_f, y_train)

    y_pred = rf.predict(Xte_f)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2 = r2_score(y_test, y_pred)

    result = permutation_importance(
        estimator=rf,
        X=Xte_f,
        y=y_test,
        scoring="neg_root_mean_squared_error",
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    permutation_results = pd.DataFrame(
        {
            "feature": Xte_f.columns,
            "permutation_importance_mean": result.importances_mean,
            "permutation_importance_std": result.importances_std,
        }
    ).sort_values("permutation_importance_mean", ascending=False).reset_index(drop=True)
    permutation_results.insert(0, "permutation_rank", range(1, len(permutation_results) + 1))

    built_in_results = pd.DataFrame(
        {
            "feature": Xtr_f.columns,
            "built_in_importance": rf.feature_importances_,
        }
    ).sort_values("built_in_importance", ascending=False).reset_index(drop=True)
    built_in_results.insert(0, "built_in_rank", range(1, len(built_in_results) + 1))

    built_in_max = built_in_results["built_in_importance"].max()
    permutation_max = permutation_results["permutation_importance_mean"].clip(lower=0).max()

    built_in_results["built_in_importance_normalized"] = (
        built_in_results["built_in_importance"] / built_in_max
        if built_in_max > 0
        else 0.0
    )
    permutation_results["permutation_importance_normalized"] = (
        permutation_results["permutation_importance_mean"].clip(lower=0) / permutation_max
        if permutation_max > 0
        else 0.0
    )

    comparison = built_in_results.merge(permutation_results, on="feature", how="inner")
    comparison["rank_difference"] = (
        comparison["built_in_rank"] - comparison["permutation_rank"]
    ).abs()
    comparison["signed_rank_change"] = (
        comparison["permutation_rank"] - comparison["built_in_rank"]
    )
    comparison["importance_gap_normalized"] = (
        comparison["built_in_importance_normalized"]
        - comparison["permutation_importance_normalized"]
    ).abs()
    comparison["comparison_status"] = comparison["rank_difference"].apply(
        classify_rank_comparison
    )
    comparison = comparison.sort_values("built_in_rank").reset_index(drop=True)

    top_builtin = built_in_results.head(10).copy()
    top_permutation = permutation_results.head(10).copy()
    built_in_top10 = set(top_builtin["feature"])
    permutation_top10 = set(top_permutation["feature"])
    shared_top10 = sorted(built_in_top10.intersection(permutation_top10))
    built_in_only = sorted(built_in_top10.difference(permutation_top10))
    permutation_only = sorted(permutation_top10.difference(built_in_top10))

    largest_disagreements = comparison.sort_values(
        "rank_difference", ascending=False
    ).head(10).copy()
    largest_disagreement = largest_disagreements.iloc[0]

    rank_correlation = comparison[["built_in_rank", "permutation_rank"]].corr(
        method="spearman"
    ).iloc[0, 1]
    negative_count = int((permutation_results["permutation_importance_mean"] < 0).sum())
    near_zero_threshold = 0.01
    near_zero_count = int(
        (
            permutation_results["permutation_importance_mean"].abs()
            <= near_zero_threshold
        ).sum()
    )

    if rank_correlation >= 0.70:
        correlation_interpretation = "The rankings show relatively strong overall agreement."
    elif rank_correlation >= 0.40:
        correlation_interpretation = "The rankings show moderate overall agreement."
    elif rank_correlation >= 0:
        correlation_interpretation = "The rankings show weak overall agreement."
    else:
        correlation_interpretation = "The rankings show an inverse overall relationship."

    built_in_table_text = top_builtin[
        ["built_in_rank", "feature", "built_in_importance"]
    ].round(4).to_string(index=False)
    permutation_table_text = top_permutation[
        [
            "permutation_rank",
            "feature",
            "permutation_importance_mean",
            "permutation_importance_std",
        ]
    ].round(4).to_string(index=False)
    disagreement_table_text = largest_disagreements[
        ["feature", "built_in_rank", "permutation_rank", "rank_difference"]
    ].to_string(index=False)

    interpretability_note = f"""SESSION 43: INTERPRETABILITY COMPARISON NOTE

Purpose
This analysis compared built-in Random Forest feature importance with test-set permutation importance. The model predicted students' final grade, G3, while G1 and G2 were excluded from the predictors to preserve the early-warning design.

Methods
Built-in importance measures how much each predictor reduced prediction error across the Random Forest's tree splits. Permutation importance was calculated on the held-out test set. Each transformed predictor was shuffled {N_REPEATS} times, and the deterioration in performance was measured using negative root mean squared error.

Main Results
The highest-ranked built-in predictor was {top_builtin.iloc[0]['feature']}.
The highest-ranked permutation predictor was {top_permutation.iloc[0]['feature']}.

The two top-10 lists shared {len(shared_top10)} predictors:
{format_feature_list(shared_top10)}.

Predictors found only in the built-in top 10:
{format_feature_list(built_in_only)}.

Predictors found only in the permutation top 10:
{format_feature_list(permutation_only)}.

The Spearman rank correlation between the complete built-in and permutation rankings was {rank_correlation:.4f}. {correlation_interpretation}

Largest Disagreement
The largest disagreement involved {largest_disagreement['feature']}. Its built-in rank was {int(largest_disagreement['built_in_rank'])}, while its permutation rank was {int(largest_disagreement['permutation_rank'])}. The absolute rank difference was {int(largest_disagreement['rank_difference'])}.

Near-Zero and Negative Values
There were {near_zero_count} predictors with absolute permutation importance no greater than {near_zero_threshold:.2f}. There were {negative_count} predictors with negative mean permutation importance. Near-zero and negative values can result from noise, sampling variation, correlated predictors, redundant information, or overfitting; they do not establish a harmful causal effect.

Interpretation of Disagreements
The methods answer different questions. Built-in importance describes how the Random Forest constructed its trees. Permutation importance evaluates how strongly held-out test performance depends on each predictor after training. Correlated predictors can share information, so shuffling one may have little effect when another still supplies similar information. One-hot encoding and differences in available split points can also affect built-in importance.

Recommendation
Permutation importance should receive greater emphasis when the objective is to understand predictors supporting performance on unseen data. Built-in importance remains useful for describing internal model use. The most convincing predictors rank highly under both methods and remain stable across repeated splits or cross-validation folds.

These importance values represent predictive associations and do not prove causation.

TOP 10 BUILT-IN IMPORTANCE RESULTS
{built_in_table_text}

TOP 10 PERMUTATION IMPORTANCE RESULTS
{permutation_table_text}

TEN LARGEST RANKING DISAGREEMENTS
{disagreement_table_text}
"""

    activity_note = f"""Session 43 Student Activity: Importance Comparison

The highest-ranked built-in predictor was {top_builtin.iloc[0]['feature']}.
The highest-ranked permutation predictor was {top_permutation.iloc[0]['feature']}.
The two top-10 lists shared {len(shared_top10)} predictors: {format_feature_list(shared_top10)}.
The largest ranking disagreement involved {largest_disagreement['feature']}, with an absolute rank difference of {int(largest_disagreement['rank_difference'])}.
The Spearman rank correlation was {rank_correlation:.4f}.

Agreement provides stronger interpretive evidence. Disagreement does not mean that either calculation is automatically wrong because the two methods measure different aspects of the fitted model. Correlation, redundancy, encoding, overfitting, and test-sample variation can change the rankings. These are predictive relationships, not causal conclusions.
"""

    permutation_results.to_csv(
        interpretation_dir / "permutation_importance.csv", index=False
    )
    top_permutation.to_csv(
        interpretation_dir / "top_10_permutation_importance.csv", index=False
    )

    comparison_output = comparison[
        [
            "feature",
            "built_in_rank",
            "permutation_rank",
            "rank_difference",
            "built_in_importance",
            "permutation_importance_mean",
            "permutation_importance_std",
            "comparison_status",
        ]
    ].copy()
    comparison_output.to_csv(
        interpretation_dir / "interpretability_comparison.csv", index=False
    )
    largest_disagreements.to_csv(
        interpretation_dir / "largest_importance_disagreements.csv", index=False
    )

    (interpretation_dir / "interpretability_comparison_note.txt").write_text(
        interpretability_note, encoding="utf-8"
    )
    (interpretation_dir / "student_activity_importance_comparison.txt").write_text(
        activity_note, encoding="utf-8"
    )

    model_metrics = pd.DataFrame(
        {
            "model": ["Random Forest Regressor"],
            "test_mae": [mae],
            "test_rmse": [rmse],
            "test_r2": [r2],
            "n_estimators": [N_ESTIMATORS],
            "random_state": [RANDOM_STATE],
            "permutation_repeats": [N_REPEATS],
            "permutation_scoring": ["neg_root_mean_squared_error"],
            "dataset_rows": [len(df)],
            "transformed_predictors": [Xtr_f.shape[1]],
            "g1_g2_excluded": [True],
        }
    )
    model_metrics.to_csv(interpretation_dir / "s43_model_metrics.csv", index=False)

    plot_perm = top_permutation.sort_values(
        "permutation_importance_mean", ascending=True
    )
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(
        plot_perm["feature"],
        plot_perm["permutation_importance_mean"],
        xerr=plot_perm["permutation_importance_std"],
        color="darkorange",
        edgecolor="black",
        alpha=0.85,
        capsize=4,
    )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Decrease in Test Performance After Shuffling")
    ax.set_ylabel("Predictor")
    ax.set_title("Top 10 Predictors by Permutation Importance")
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    fig.tight_layout()
    fig.savefig(figures_dir / "permutation_importance.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    features_to_plot = built_in_top10.union(permutation_top10)
    plot_data = comparison[comparison["feature"].isin(features_to_plot)].copy()
    plot_data = plot_data.sort_values("permutation_importance_normalized")
    y_positions = np.arange(len(plot_data))
    bar_height = 0.38

    fig, ax = plt.subplots(figsize=(12, max(7, len(plot_data) * 0.55)))
    ax.barh(
        y_positions - bar_height / 2,
        plot_data["built_in_importance_normalized"],
        height=bar_height,
        label="Built-in importance",
        color="steelblue",
        edgecolor="black",
    )
    ax.barh(
        y_positions + bar_height / 2,
        plot_data["permutation_importance_normalized"],
        height=bar_height,
        label="Permutation importance",
        color="darkorange",
        edgecolor="black",
    )
    ax.set_yticks(y_positions)
    ax.set_yticklabels(plot_data["feature"])
    ax.set_xlabel("Normalized Importance")
    ax.set_ylabel("Predictor")
    ax.set_title("Built-In Versus Permutation Importance")
    ax.legend()
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    fig.tight_layout()
    fig.savefig(
        figures_dir / "built_in_vs_permutation_importance.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    summary_text = "\n".join(
        [
            "SESSION 43 RESULTS",
            "=" * 68,
            f"Rows analyzed: {len(df)}",
            f"Original predictors: {X.shape[1]}",
            f"Transformed predictors: {Xtr_f.shape[1]}",
            "Leakage check: PASSED - G1 and G2 excluded",
            f"Test MAE: {mae:.4f}",
            f"Test RMSE: {rmse:.4f}",
            f"Test R2: {r2:.4f}",
            f"Top built-in predictor: {top_builtin.iloc[0]['feature']}",
            f"Top permutation predictor: {top_permutation.iloc[0]['feature']}",
            f"Shared top-10 predictors: {len(shared_top10)}",
            f"Spearman rank correlation: {rank_correlation:.4f}",
            "Optional SHAP extension: not required for Section 8",
            "=" * 68,
        ]
    )

    return {
        "summary_text": summary_text,
        "interpretability_note": interpretability_note,
        "top_permutation_text": permutation_table_text,
        "top_builtin_text": built_in_table_text,
        "metrics": {
            "mae": float(mae),
            "rmse": float(rmse),
            "r2": float(r2),
            "rank_correlation": float(rank_correlation),
        },
    }


RESULTS = None
if __name__ == "__main__":
    RESULTS = run_analysis()


SESSION 43 RESULTS
Rows analyzed: 395
Original predictors: 30
Transformed predictors: 56
Leakage check: PASSED - G1 and G2 excluded
Test MAE: 3.0095
Test RMSE: 3.7613
Test R2: 0.3100
Top built-in predictor: absences
Top permutation predictor: failures
Shared top-10 predictors: 5
Spearman rank correlation: 0.4256
Optional SHAP extension: not required for Section 8


## Calculated Session 43 results

The analysis was executed locally from VS Code with `student-mat.csv`.
The early-warning leakage check excluded `G1` and `G2` before model training.

```text
SESSION 43 RESULTS
====================================================================
Rows analyzed: 395
Original predictors: 30
Transformed predictors: 56
Leakage check: PASSED - G1 and G2 excluded
Test MAE: 3.0095
Test RMSE: 3.7613
Test R2: 0.3100
Top built-in predictor: absences
Top permutation predictor: failures
Shared top-10 predictors: 5
Spearman rank correlation: 0.4256
Optional SHAP extension: not required for Section 8
====================================================================
```

### Top built-in importance results

```text
 built_in_rank    feature  built_in_importance
             1   absences               0.1878
             2   failures               0.1460
             3     health               0.0493
             4      goout               0.0462
             5        age               0.0371
             6  studytime               0.0321
             7   freetime               0.0290
             8 traveltime               0.0277
             9       Fedu               0.0269
            10       Walc               0.0257
```

### Top permutation importance results

```text
 permutation_rank        feature  permutation_importance_mean  permutation_importance_std
                1       failures                       0.7211                      0.1818
                2       absences                       0.4676                      0.1511
                3          goout                       0.1477                      0.0487
                4            age                       0.0855                      0.0355
                5           Medu                       0.0755                      0.0364
                6          sex_M                       0.0561                      0.0193
                7          sex_F                       0.0549                      0.0277
                8      studytime                       0.0548                      0.0371
                9   Mjob_at_home                       0.0524                      0.0343
               10 guardian_other                       0.0464                      0.0393
```


## Interpretation note

SESSION 43: INTERPRETABILITY COMPARISON NOTE

Purpose
This analysis compared built-in Random Forest feature importance with test-set permutation importance. The model predicted students' final grade, G3, while G1 and G2 were excluded from the predictors to preserve the early-warning design.

Methods
Built-in importance measures how much each predictor reduced prediction error across the Random Forest's tree splits. Permutation importance was calculated on the held-out test set. Each transformed predictor was shuffled 10 times, and the deterioration in performance was measured using negative root mean squared error.

Main Results
The highest-ranked built-in predictor was absences.
The highest-ranked permutation predictor was failures.

The two top-10 lists shared 5 predictors:
absences, age, failures, goout, studytime.

Predictors found only in the built-in top 10:
Fedu, Walc, freetime, health, traveltime.

Predictors found only in the permutation top 10:
Medu, Mjob_at_home, guardian_other, sex_F, sex_M.

The Spearman rank correlation between the complete built-in and permutation rankings was 0.4256. The rankings show moderate overall agreement.

Largest Disagreement
The largest disagreement involved health. Its built-in rank was 3, while its permutation rank was 56. The absolute rank difference was 53.

Near-Zero and Negative Values
There were 27 predictors with absolute permutation importance no greater than 0.01. There were 16 predictors with negative mean permutation importance. Near-zero and negative values can result from noise, sampling variation, correlated predictors, redundant information, or overfitting; they do not establish a harmful causal effect.

Interpretation of Disagreements
The methods answer different questions. Built-in importance describes how the Random Forest constructed its trees. Permutation importance evaluates how strongly held-out test performance depends on each predictor after training. Correlated predictors can share information, so shuffling one may have little effect when another still supplies similar information. One-hot encoding and differences in available split points can also affect built-in importance.

Recommendation
Permutation importance should receive greater emphasis when the objective is to understand predictors supporting performance on unseen data. Built-in importance remains useful for describing internal model use. The most convincing predictors rank highly under both methods and remain stable across repeated splits or cross-validation folds.

These importance values represent predictive associations and do not prove causation.

TOP 10 BUILT-IN IMPORTANCE RESULTS
 built_in_rank    feature  built_in_importance
             1   absences               0.1878
             2   failures               0.1460
             3     health               0.0493
             4      goout               0.0462
             5        age               0.0371
             6  studytime               0.0321
             7   freetime               0.0290
             8 traveltime               0.0277
             9       Fedu               0.0269
            10       Walc               0.0257

TOP 10 PERMUTATION IMPORTANCE RESULTS
 permutation_rank        feature  permutation_importance_mean  permutation_importance_std
                1       failures                       0.7211                      0.1818
                2       absences                       0.4676                      0.1511
                3          goout                       0.1477                      0.0487
                4            age                       0.0855                      0.0355
                5           Medu                       0.0755                      0.0364
                6          sex_M                       0.0561                      0.0193
                7          sex_F                       0.0549                      0.0277
                8      studytime                       0.0548                      0.0371
                9   Mjob_at_home                       0.0524                      0.0343
               10 guardian_other                       0.0464                      0.0393

TEN LARGEST RANKING DISAGREEMENTS
        feature  built_in_rank  permutation_rank  rank_difference
         health              3                56               53
  Mjob_services             14                55               41
           Fedu              9                45               36
           Dalc             18                54               36
guardian_mother             53                22               31
    reason_home             24                52               28
      address_R             55                28               27
guardian_father             41                15               26
          sex_F             30                 7               23
    Mjob_health             39                17               22
